# 第16章 速く・大きく：推論と学習の工学

KVキャッシュ、賢いサイコロ、メモリの節約術、そして何百枚ものGPU

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：KVキャッシュ付きの生成（出力が変わらず速くなることを確認）、top-k / top-p サンプリング、勾配累積、FlashAttention の検証
- **身につく数学・理論**：KVキャッシュ：計算の使い回し ／ KVキャッシュのメモリとGQA（グループ化クエリAttention） ／ top-k・top-p（nucleus）・くり返しペナルティ ／ 勾配累積：小さなGPUで大きなバッチ ／ 活性化の再計算（勾配チェックポイント） ／ FlashAttention（メモリに優しいAttention） ／ 分散学習：データ並列・テンソル並列・パイプライン並列・ZeRO
- **新しく使うPython・PyTorch**：辞書やリストに途中結果をためる ／ `time.perf_counter` で速さを測る ／ `F.scaled_dot_product_attention` ／ `torch.utils.checkpoint`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> GPTの「仕組み」は前の章で完成しました。この章は「**工学**」の章です。同じGPTを、何倍も速く動かし、少ないメモリで学習させるための技を、実際にコードで確かめていきます。どれも、実際のLLMの開発現場で毎日使われている技です。

> **🔗 これまでの復習とつながり：第3章・第7章・第11章とのつながり**
>
> - **第3章（確率と温度・自己回帰生成）**：温度だけで多様性を上げると「ありえない単語」が混ざる問題と、1トークン生成するたびに文章全体を毎回モデルに通し直す「遅さ」の正体を見ました。
> - **第7章（自動微分と `.grad`）**：PyTorchでは `backward()` を呼ぶたびに `.grad` が足し算（累積）される性質と、逆伝播のために順伝播の途中結果（活性化）をメモリに保存する仕組みを学びました。
> - **第11章（因果Attention $QK^\top$）**：因果マスクにより過去のトークンは未来のトークンを一切見ないこと、そして $T \times T$ のスコア表が文章の長さの2乗で重くなることを確認しました。
>
> この章では、その3つの知識をすべて逆手にとって、**KVキャッシュ・GQA・top-k/top-p・勾配累積・勾配チェックポイント・FlashAttention** という実務の必須工学へとつなげます。

**ミナト**：仕組みが分かったのに、まだ学ぶことがあるんですか？

**ボンゴ先生**：車の仕組みが分かっても、F1で勝てるとは限らない。ChatGPTが何億人もの質問に答えられるのは、モデルが賢いからだけじゃない。同じ計算を、いかに速く・安く・たくさんこなすか。その工夫の積み重ねなんだ。今日はピットクルーの修行だよ。

## 準備：キャッシュ対応のGPT

> **✨ コツ：実行環境とリソースの目安**
>
> - **環境**：CPUで完走できます（GPU不要・外部ダウンロードなし）。
> - **メモリと所要時間**：必要RAMは1GB未満、すべての実験を合わせても数秒〜数十秒で終わります。
> - **GPUで試す場合**：下のコードは `device=x.device` / `device=idx.device` に対応しているため、`model.to("cuda")` してGPU上で速度差を測ることもできます。

第15章のGPT-2形式のAttentionに、**KVキャッシュ**という仕組みを追加したものを用意します。重みはでたらめでかまいません（速さと正しさを確かめるだけなので）。

> **✨ コツ：この章のコードは「動かして、結果を見る」でOK**
>
> この章のコードは、これまでより密度が高めです。全部の行を理解できなくても大丈夫。まずは貼り付けて実行し、「結果が同じで、速くなった」「勾配が同じ」といった**結果**を確かめましょう。仕組みは「理論の小部屋」で説明します。気になった行があれば、`print(…shape)` を入れて中身をのぞけばOKです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

class CachedAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, hs).transpose(1, 2) for t in (q, k, v)]
        if cache is not None:                      # 過去のKとVを使い回す
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        L = k.shape[2]                             # 過去も含めた長さ
        att = q @ k.transpose(-2, -1) / math.sqrt(hs)
        mask = torch.tril(torch.ones(L, L, device=x.device))[L - T:]    # 新しいT個の行だけのマスク
        att = att.masked_fill(mask == 0, float("-inf"))
        y = F.softmax(att, dim=-1) @ v
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class TinyGPT(nn.Module):
    def __init__(self, V=1000, n_embd=256, n_head=4, n_layer=6, block_size=512):
        super().__init__()
        self.wte, self.wpe = nn.Embedding(V, n_embd), nn.Embedding(block_size, n_embd)
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layer)])
        self.attn = nn.ModuleList([CachedAttention(n_embd, n_head) for _ in range(n_layer)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layer)])
        self.mlp = nn.ModuleList([nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd)) for _ in range(n_layer)])
        self.ln_f, self.head = nn.LayerNorm(n_embd), nn.Linear(n_embd, V)

    def forward(self, idx, caches=None, start_pos=0):
        T = idx.shape[1]
        x = self.wte(idx) + self.wpe(torch.arange(start_pos, start_pos + T, device=idx.device))
        for i in range(len(self.attn)):
            x = x + self.attn[i](self.ln1[i](x), None if caches is None else caches[i])
            x = x + self.mlp[i](self.ln2[i](x))
        return self.head(self.ln_f(x))

torch.manual_seed(0)
model = TinyGPT().eval()
print(f"パラメータ：{sum(p.numel() for p in model.parameters()):,}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
パラメータ：5,383,144
```

</details>

## ムダに気づく：生成のたびに全部計算し直している

第3章からずっと使ってきた生成ループを思い出してください。

> 文章全体をモデルに通す → 最後の位置の確率で1トークン選ぶ → 付け足す → **また文章全体をモデルに通す** …

100トークン目を生成するとき、1〜99トークン目の計算を、またゼロからやり直しています。でも、因果マスクのおかげで、**過去の位置の計算結果は、未来のトークンが増えても変わりません**（第11章のカンニング検査で確かめましたね）。だったら、前回の計算結果を取っておいて使い回せばいいはずです。

> **💡 理論のポイント：KVキャッシュ**
>
> Attentionで、過去の位置から受け取るのは **Key** と **Value** でした。過去の位置のKeyとValueは、一度計算したら変わりません。そこで、
>
> - 過去のトークンのK・Vを、層ごとに **キャッシュ**（取っておく場所）に保存しておく
> - 新しいトークンが来たら、**その1トークンぶんだけ**Q・K・Vを計算し、Kと Vをキャッシュの後ろに付け足す
> - 新しいトークンのQで、キャッシュの全K・Vに対してAttentionを計算する
>
> こうすると、1トークンを生成するための計算が「文章全体」から「新しく生まれた1トークンぶん」に減ります。もしキャッシュがないと、$N$ 個のトークンを出すのに毎回 $1 + 2 + \dots + N \approx N^2/2$ トークンぶんの重複計算をしてしまいますが、キャッシュがあれば過去の計算をそのまま再利用できるため、生成が劇的に速くなります。上のコードの `cache` がそれです。これを **KVキャッシュ** と言い、ほぼすべてのLLMの推論で使われています。

キャッシュなし・ありの2通りで生成して、**結果が同じで、速さが違う**ことを確かめましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def generate_naive(model, idx, n):
    for _ in range(n):
        logits = model(idx)                              # 毎回、全体を計算
        idx = torch.cat([idx, logits[:, -1:].argmax(-1)], dim=1)
    return idx

@torch.no_grad()
def generate_cached(model, idx, n):
    caches = [{} for _ in model.attn]
    logits = model(idx, caches)                          # 最初だけ全体を計算
    pos = idx.shape[1]
    for _ in range(n):
        next_id = logits[:, -1:].argmax(-1)
        idx = torch.cat([idx, next_id], dim=1)
        logits = model(next_id, caches, start_pos=pos)   # 新しい1トークンだけ
        pos += 1
    return idx

prompt = torch.randint(0, 1000, (1, 20))
# time.perf_counter() は「今の時刻（秒）」。前後の差が、かかった時間になる
t0 = time.perf_counter(); a = generate_naive(model, prompt, 200); t1 = time.perf_counter()
b = generate_cached(model, prompt, 200); t2 = time.perf_counter()
print("結果は同じ？", torch.equal(a, b))
print(f"キャッシュなし {t1 - t0:.2f}秒 / キャッシュあり {t2 - t1:.2f}秒 → {(t1 - t0) / (t2 - t1):.1f}倍速い")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
結果は同じ？ True
キャッシュなし 1.93秒 / キャッシュあり 0.31秒 → 6.2倍速い
```

</details>

（時間は実行する環境によって変わります。「結果は同じ？ True」が大事です。）文章が長くなるほど、差は大きくなります。

<details><summary><b>基礎の補足：KVキャッシュの形（shape）の成長を追う</b>（クリックで開く）</summary>

`TinyGPT`（4ヘッド・ヘッド次元64）に長さ20のプロンプトを渡し、1トークンずつ生成するときの形を追ってみましょう。

| ステップ | モデルへの入力 `idx` | 新しい `q` の形 | キャッシュ `cache["k"]`, `cache["v"]` の形 | `start_pos` | Attentionスコア `att` |
|---|---|---|---|---|---|
| 最初（Prefill） | `[1, 20]` | `[1, 4, 20, 64]` | `[1, 4, 20, 64]`（新規保存） | `0` | `[1, 4, 20, 20]` |
| 1文字目生成 | `[1, 1]`（新1語のみ） | `[1, 4, 1, 64]` | `[1, 4, 21, 64]`（`dim=2` に連結） | `20` | `[1, 4, 1, 21]` |
| 2文字目生成 | `[1, 1]`（新1語のみ） | `[1, 4, 1, 64]` | `[1, 4, 22, 64]`（さらに +1） | `21` | `[1, 4, 1, 22]` |

- **なぜ過去のQuery（Q）は保存しないのか？**：位置 $t$ の出力は $\text{softmax}(q_t K_{1:t}^\top / \sqrt{d}) V_{1:t}$ で決まるため、過去の $q_1 \dots q_{t-1}$ は二度と使いません。
- **なぜ `start_pos` が必須なのか？**：デコード時は入力が常に長さ1（`T=1`）なので、`start_pos` を渡さないと全トークンに0番目の位置埋め込み `wpe(0)` が足されて出力が壊れてしまいます。

</details>

#### 📝 確認問題（`ch16-q01` · Lv.2 · 因果マスクによりKVキャッシュが数学的に一致し高速化できる理由）

KVキャッシュを使うと、素朴な生成ループと**まったく同じ出力トークン**を出しながら何倍も速くなります。過去のトークンのKeyとValueを再計算せずに使い回しても結果が変わらない根拠は何ですか？

- **a.** 過去のK・Vは新トークンごとに少しずつ変化するが、誤差が小さいため無視して使い回す近似手法だから
- **b.** 推論ループのたびにTransformerの重みを再初期化して、過去のK・Vが毎回同じ基準値へ戻る仕組みだから
- **c.** 新しく生成されたトークンのAttention計算では、過去のトークン列を一切参照せず独立に処理するから
- **d.** 因果マスクにより過去トークンは未来を参照しないため、後ろに新トークンが増えても過去のK・Vが不変だから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第11章で行った「後ろのトークンを書き換えても、前の位置の計算結果が変わらない」カンニング検査を思い出しましょう。
- **ヒント 2**：因果マスク（下三角行列）があるとき、位置 $t$ の隠れ状態やKey・Valueが位置 $t+1$ 以降に依存するかどうかを考えてみましょう。
- **正解**：`d`

**考え方**：因果マスクがあるため、位置 $t$ の隠れ状態とK・Vは位置 $1 \dots t$ だけに依存します。したがって位置 $t+1$ 以降が追加されても過去のK・Vは不変であり、安全に使い回せます。

1. 因果マスク（下三角行列）により、過去トークンは未来トークンを参照できない。
1. よってトークンが増えても過去位置のKey・Valueは完全に不変。
1. 新トークン1個分だけQ・K・Vを計算してキャッシュに連結すれば、出力は完全一致しつつ重複計算 $1 + 2 + \dots + N \approx N^2/2$ を削れる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q01

</details>

#### 📝 確認問題（`ch16-q02` · Lv.3 · KVキャッシュで過去のQuery（Q）を保存しない理由）

KVキャッシュは「Key」と「Value」だけを保存し、過去の「**Query（Q）**」は保存しません。過去のQueryを保存しなくてよい理由は何ですか？

- **a.** QueryはKeyやValueより特徴次元が何倍も大きく設計されており、キャッシュするとGPUメモリに入り切らないから
- **b.** 因果マスクを適用するとAttentionの式からQueryの項が消え、KeyとValueの内積だけで次トークンが求まるから
- **c.** 過去の出力は計算済みで、新トークンの予測に使うのは「新Query」と「過去を含む全Key・Value」だけだから
- **d.** Queryベクトルはどの位置のトークンでも常に同じ定数になるため、わざわざキャッシュへ保存する意味がないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Attentionにおける「質問する側（Query）」と「検索される側（Key・Value）」の役割分担を思い出しましょう。
- **ヒント 2**：時刻 $t$ で新しく出力を求める式 $\text{softmax}(q_t K_{1:t}^\top / \sqrt{d}) V_{1:t}$ の中に、過去の $q_1, \dots, q_{t-1}$ が現れるか確認しましょう。
- **正解**：`c`

**考え方**：新トークン $t$ の出力を求める式は $\text{softmax}(q_t K_{1:t}^\top / \sqrt{d}) V_{1:t}$ であり、$q_1, \dots, q_{t-1}$ は式の中に一切現れません。

1. Attentionは「Queryが過去のKeyと照合し、Valueを取り出す」仕組み。
1. ステップ $t$ で新しい出力が必要なのは位置 $t$ だけ → 必要なのは $q_t$ のみ。
1. 照合対象は過去全部なので $K_{1:t}$ と $V_{1:t}$ は必要だが、過去の $q_1 \dots q_{t-1}$ は二度と使わない。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q02

</details>

*(この節の追加問題 `ch16-q03` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/16-scale.html で自動採点・復習できます)*

## KVキャッシュのメモリと、GQA

キャッシュは速さと引き換えに **メモリ** を使います。どれくらい使うのか、計算してみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def kv_cache_bytes(n_layer, n_kv_head, head_dim, n_tokens, bytes_per_num=2):
    return 2 * n_layer * n_kv_head * head_dim * n_tokens * bytes_per_num   # 2 = KとV

# Qwen2.5-0.5B：24層、Queryのヘッド14、K・Vのヘッド2、ヘッドの大きさ64
per_token = kv_cache_bytes(24, 2, 64, 1)
print(f"1トークンあたり {per_token / 1024:.0f} KB → 32,768トークンで {kv_cache_bytes(24, 2, 64, 32768) / 1e9:.2f} GB")
print(f"もしK・Vのヘッドも14個なら → 32,768トークンで {kv_cache_bytes(24, 14, 64, 32768) / 1e9:.2f} GB")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
1トークンあたり 12 KB → 32,768トークンで 0.40 GB
もしK・Vのヘッドも14個なら → 32,768トークンで 2.82 GB
```

</details>

> **💡 理論のポイント：GQA：Key・Valueのヘッドを共有する**
>
> Qwen2.5-0.5Bは、Queryのヘッドが14個あるのに、KeyとValueのヘッドは**2個だけ**です。7個のQueryヘッドが、1組のKey・Valueを**共有**しているのです。これを **GQA**（Grouped-Query Attention、グループ化クエリAttention）と言います。
>
> 性能はほとんど落ちずに、KVキャッシュのメモリが7分の1になります（上の計算では2.82GB → 0.40GB）。長い文章を扱うLLMや、たくさんの人が同時に使うサービスでは、KVキャッシュのメモリが一番の悩みの種なので、最近のLLMのほとんどがGQAを採用しています。K・Vのヘッドを1組だけにした極端な版は **MQA**（Multi-Query Attention）と呼ばれます。

<details><summary><b>基礎の補足：数式の解剖：なぜKVキャッシュは $2 \times L \times H_{kv} \times d \times T \times b$ になるのか？</b>（クリックで開く）</summary>

キャッシュに保存するテンソルは、全 $L$ 層それぞれに「Key用とValue用の **2つ**」、形 `[H_kv, T, d]`、1数値あたり $b$ バイト（fp16なら $b=2$）です。したがって、

$$\text{KVキャッシュ（バイト）} = \underbrace{2}_{\text{KとV}} \times \underbrace{L}_{\text{層数}} \times \underbrace{H_{kv}}_{\text{KVヘッド数}} \times \underbrace{d}_{\text{ヘッド次元}} \times \underbrace{T}_{\text{トークン数}} \times \underbrace{b}_{\text{バイト数}}$$

| 方式 | Queryヘッド数 $H_q$ | KVヘッド数 $H_{kv}$ | KVキャッシュの大きさ | 特徴 |
|---|---|---|---|---|
| **MHA**（従来型） | 14 | 14 | 大きい（基準 1倍） | 品質は高いが長文でVRAMを圧迫 |
| **GQA**（現代の標準） | 14 | 2（7個ずつ共有） | **1/7 に削減！** | 品質をほぼ保ちつつ大幅省メモリ |
| **MQA**（極端版） | 14 | 1（全部で共有） | 1/14（最小） | 最省メモリだがやや品質低下することも |

</details>

#### 📝 確認問題（`ch16-q04` · Lv.3 · KVキャッシュのメモリ計算式 2 × L × H_kv × d × T × b）

KVキャッシュのメモリ量は **$2 \times L \times H_{kv} \times d \times T \times b$** バイトで計算できます（$2$ はKとV、$L$ は層数、$H_{kv}$ はKVヘッド数、$d$ はヘッド次元、$T$ はトークン数、$b$ は1数値のバイト数）。Qwen2.5-0.5B（$L=24, H_{kv}=2, d=64, b=2$ バイト＝fp16）のとき、**1トークンあたり（$T=1$）** 何 **KB**（1KB = 1,024バイト）使いますか？（整数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：問題文の式 $2 \times L \times H_{kv} \times d \times T \times b$ で1トークンあたりのバイト数を求め、最後に1KB＝1,024バイトで割ります。
- **ヒント 2**：$L=24, H_{kv}=2, d=64, T=1, b=2$ をすべて掛け合わせて総バイト数を出し、その値を $1{,}024$ で割りましょう。
- **正解**：`12` KB

**考え方**：$2 \times 24 \times 2 \times 64 \times 2 = 12{,}288$ バイト、$12{,}288 \div 1{,}024 = 12$ KB です。

1. KとVの2つ × 24層 × 2 KVヘッド × 64次元 × 2バイト。
1. $2 \times 24 \times 2 \times 64 \times 2 = 12{,}288$ バイト。
1. $12{,}288 \div 1{,}024 = 12$ KB / トークン。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q04

</details>

#### 📝 確認問題（`ch16-q05` · Lv.2 · MHA・GQA・MQAの違いとGQAが標準になった理由）

現代のLLMの多くが採用している **GQA（Grouped-Query Attention）** の仕組みとして正しいものはどれですか？

- **a.** 語彙テーブルを意味の近いグループへ分割して管理し、入力文章のトークン列の長さを約半分に圧縮する
- **b.** すべてのQueryヘッドに対してそれより多いKey・Valueヘッドを割り当て、表現力と推論精度を引き上げる
- **c.** メモリを節約するためにKeyとValueの投影層を完全に廃止し、Query同士の内積だけでAttentionを行う
- **d.** 複数のQueryヘッドをグループ分けし、各グループで1組のK・Vを共有してKVキャッシュを削減する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：GQA（Grouped-Query Attention）の「Grouped」は何をグループ化して何を共有しているのか思い出しましょう。
- **ヒント 2**：Qwen2.5-0.5BではQueryヘッド14個に対し、KVヘッドは2個（7個ずつのグループで共有）でした。これによりどのメモリが減るか確認しましょう。
- **正解**：`d`

**考え方**：MHA（全ヘッド独立：高品質だがKVキャッシュ巨大）とMQA（KVを1組に統一：超省メモリだが少し品質低下）のちょうど中間をとるのがGQAです。

1. MHA：$H_{kv} = H_q$（メモリ大・品質高）。
1. GQA：$1 < H_{kv} < H_q$（メモリ大幅削減・品質ほぼ維持）。
1. MQA：$H_{kv} = 1$（メモリ最小・やや品質低下することがある）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q05

</details>

## 賢いサイコロ：top-k と top-p

第3章では、温度でサイコロの尖り具合を調整しました。でも、温度を上げると「ありえない候補」まで選ばれてしまう問題がありました（温度1.5の支離滅裂な文章を思い出してください）。そこで、**候補をあらかじめ絞ってから**サイコロを振る方法が使われます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
words = ["猫", "犬", "人間", "名前", "机", "宇宙", "の"]
logits = torch.tensor([3.0, 1.6, 1.2, 0.4, -1.0, -1.5, -2.0])
probs = F.softmax(logits, dim=0)

def top_k_filter(probs, k):
    keep = torch.zeros_like(probs)
    idx = torch.topk(probs, k).indices
    keep[idx] = probs[idx]
    return keep / keep.sum()

def top_p_filter(probs, p):
    sorted_p, order = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=0)
    n_keep = int((cum < p).sum()) + 1            # 累積がpを超えるまでの最小の候補数
    keep = torch.zeros_like(probs)
    keep[order[:n_keep]] = probs[order[:n_keep]]
    return keep / keep.sum()

show = lambda p: "  ".join(f"{w}{x:.2f}" for w, x in zip(words, p) if x > 0)
print("そのまま ：", show(probs))
print("top-k=3 ：", show(top_k_filter(probs, 3)))
print("top-p=0.9：", show(top_p_filter(probs, 0.9)))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
そのまま ： 猫0.66  犬0.16  人間0.11  名前0.05  机0.01  宇宙0.01  の0.00
top-k=3 ： 猫0.71  犬0.17  人間0.12
top-p=0.9： 猫0.71  犬0.17  人間0.12
```

</details>

> **💡 理論のポイント：top-k と top-p（nucleus sampling）**
>
> - **top-k**：確率の高い上位 **k個** だけを機械的に候補に残す方法です。「テストの成績上位k人を常に合格にする」定員固定の選抜のようなものです。シンプルですが、1位の候補が確率99%で確定している場面でも無理やりk個の候補を残してしまい、たまに事故が起きます。
> - **top-p**（nucleus sampling、核サンプリング）：確率の高い順に足していき、合計が **p**（例えば0.9＝90%）を超えるまでの候補だけを残す方法です。「上位から足して得点全体の9割を占めるグループを選ぶ」方式です。モデルが答えに自信満々なときは1〜2個でスッと打ち切られ、迷っているときは自然に多くの候補が残るため、状況に応じて賢く柔軟に絞り込めます。
>
> どちらも、残した候補の確率を足して1になるように割り直してから、サイコロを振ります。実際のチャットでは、「温度0.7、top-p 0.8」のような組み合わせがよく使われます。
>
> もう1つ、同じ言葉をくり返しすぎるのを防ぐ **くり返しペナルティ**（repetition penalty）もよく使われます。すでに出てきたトークンの点数を少し下げる、という単純な方法です。第3章で小さなモデルが同じフレーズをくり返したのを、これで和らげられます。

<details><summary><b>基礎の補足：5語の手計算で完全理解：`top-k=2` と `top-p=0.85` と再正規化</b>（クリックで開く）</summary>

5つの単語の確率が `["猫": 0.50, "犬": 0.30, "鳥": 0.10, "机": 0.05, "月": 0.05]` だったとします。

| 順位 | 単語 | 元の確率 | 累積確率 | `top-k = 2`（上位2個） | `top-p = 0.85`（累積が0.85以上になる最小個数） |
|---|---|---|---|---|---|
| 1 | 猫 | `0.50` | `0.50` | 残す → $0.50 / 0.80 = \mathbf{0.625}$ | 残す → $0.50 / 0.90 = \mathbf{0.556}$ |
| 2 | 犬 | `0.30` | `0.80`（まだ0.85未満！） | 残す → $0.30 / 0.80 = \mathbf{0.375}$ | 残す → $0.30 / 0.90 = \mathbf{0.333}$ |
| 3 | 鳥 | `0.10` | **`0.90`**（ここで0.85超え！） | 0 | 残す → $0.10 / 0.90 = \mathbf{0.111}$ |
| 4 | 机 | `0.05` | `0.95` | 0 | 0 |
| 5 | 月 | `0.05` | `1.00` | 0 | 0 |

ポイントは2つです。① `top-p = 0.85` では2語目の累積が `0.80 < 0.85` なので**3語目まで含めて3個残る**こと、② 切り落とした後に残った合計（`0.80` や `0.90`）で割り直す（**再正規化**）ことで、確率の合計を `1.0` に戻すことです。

</details>

*(対話デモはWeb教科書ページで確認できます：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html)*

上のデモは、第3章のサイコロに top-k と top-p のつまみを付けたものです。温度を上げて候補を平らにしてから、top-p を0.8くらいに下げてみましょう。「机」のようなありえない候補が、サイコロから消えるのが分かります。`transformers` では、次のように指定するだけで使えます（Qwenなどで試すときに）。

```python
out = model.generate(**inputs, max_new_tokens=100, do_sample=True,
                     temperature=0.7, top_p=0.8, top_k=50, repetition_penalty=1.1)
```

#### 📝 確認問題（`ch16-q06` · Lv.3 · top-p（nucleus sampling）で残る候補数の手計算）

5つの単語の予測確率が **`[0.50, 0.30, 0.10, 0.05, 0.05]`** でした。**`top-p = 0.85`** を適用すると、サイコロの候補に残る単語は何個になりますか？（整数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：top-p（nucleus sampling）では、確率の高い候補から順に累積和を足し、初めてしきい値 $p$ 以上になる最小の候補集合を残します。
- **ヒント 2**：1位までで `0.50`、2位まで足すと `0.50 + 0.30 = 0.80` です。しきい値 `0.85` に届いているか確認し、足りなければ次の候補も足して個数を数えましょう。
- **正解**：`3` 個

**考え方**：累積確率が初めて $p = 0.85$ 以上になる最小の候補集合を残すため、2個目（0.80）では足りず、3個目（0.90）まで含めた **3個** が残ります。

1. 1位まで：`0.50`（$< 0.85$）。
1. 2位まで：`0.50 + 0.30 = 0.80`（$< 0.85$）。
1. 3位まで：`0.80 + 0.10 = 0.90`（$\ge 0.85$ → ここで確定！）。
1. 残る候補は **3個**（合計0.90で割って再正規化すると `[0.556, 0.333, 0.111]` になる）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q06

</details>

#### 📝 確認問題（`ch16-q07` · Lv.3 · top-k と top-p の決定的な違いと再正規化）

ある文脈ではモデルが自信満々で1位が確率 **0.98**、別の文脈では迷っていて上位10語が **0.08ずつ** 並んでいます。この2つの状況に応じて**残す候補数を自動で1個と10個に柔軟に切り替えられる**のはどちらの方法ですか？

- **a.** どちらも固定人数の候補だけを残すため、文脈に応じて個数を切り替えることはできない
- **b.** **top-k**（上位k個を選ぶ設定が確率分布に応じて自動でkを増減させる）
- **c.** **くり返しペナルティ**（既出トークンを除外して残る候補数を自動調整する）
- **d.** **top-p**（累積確率のしきい値で切るため、分布の尖り具合で候補数が変わる）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「常に上位 $k$ 個で定員固定する方式」と「累積確率がしきい値に達するまで上から採用する方式」の違いを比べましょう。
- **ヒント 2**：1位だけで確率 `0.98` ある場面と、`0.08` が10語並ぶ場面それぞれで、累積和が `0.9` に達する個数を想像してみましょう。
- **正解**：`d`

**考え方**：top-p（nucleus sampling）は確率の質量（累積和）を基準にするため、確信度が高い場面での暴走を防ぎつつ、多様性が欲しい場面では豊富な候補を保てます。

1. 自信満々（1位 0.98）＋ `top-p = 0.9` → 1位だけで 0.98 ≥ 0.9 → 1個だけ残る。
1. 迷っている（各0.08が10語＋α）＋ `top-p = 0.9` → 累積が0.9を超えるまで12個ほど残る。
1. このように分布の尖り具合に自動適応するのが top-p の最大の強み。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q07

</details>

*(この節の追加問題 `ch16-q08`, `ch16-q09` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/16-scale.html で自動採点・復習できます)*

## 学習の工学1：勾配累積

T4のメモリに、バッチ64が入らない。でもバッチ64で学習したい。そんなときの技が **勾配累積**（gradient accumulation）です。小さなバッチで何回か `backward()` して勾配を**ためて**から、まとめて1回更新します。第7章で「`.grad` は足し算されていく」と注意しましたが、それを逆に利用するのです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
net = nn.Linear(16, 4)
x, y = torch.randn(32, 16), torch.randint(0, 4, (32,))

net.zero_grad()
F.cross_entropy(net(x), y).backward()            # バッチ32で1回
g_big = net.weight.grad.clone()

net.zero_grad()
for i in range(4):                               # バッチ8を4回に分ける
    xb, yb = x[i * 8:(i + 1) * 8], y[i * 8:(i + 1) * 8]
    (F.cross_entropy(net(xb), yb) / 4).backward()    # 4で割ってから足し込む
g_acc = net.weight.grad

print("勾配は同じ？", torch.allclose(g_big, g_acc, atol=1e-6))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
勾配は同じ？ True
```

</details>

メモリに乗るのは「バッチ8」ぶんだけなのに、バッチ32で計算したのと同じ勾配が手に入りました。計算の量はバッチ32を1回やるのと同じなので、時間は大きくは変わりません（小分けにするぶん、少し遅くなる程度）。減るのは**途中の計算結果（活性化）のメモリ**で、だいたい4分の1になります。モデルの重み・勾配・AdamWの記録（第15章の「1パラメータあたり16バイト」）は、バッチの大きさに関係なく必要なので減りません。損失を累積回数（4）で割るのを忘れないのがコツです。

<details><summary><b>基礎の補足：なぜ「損失 ÷ `accum_steps`」が必須なのか？4サンプルで確かめる</b>（クリックで開く）</summary>

`F.cross_entropy` はバッチ内の**平均**を返します。4サンプルの損失 $\ell_1, \ell_2, \ell_3, \ell_4$ を「バッチ4一括」と「バッチ2 × 2回累積」で比べてみましょう。

$$\text{バッチ4一括の平均} = \frac{\ell_1 + \ell_2 + \ell_3 + \ell_4}{4} = \frac{1}{2}\left(\frac{\ell_1 + \ell_2}{2}\right) + \frac{1}{2}\left(\frac{\ell_3 + \ell_4}{2}\right)$$

それぞれのミニバッチ平均を `accum_steps`（ここでは `2`）で割ってから足し込むと、一括平均と**数学的に完全一致**します。もし割り忘れると勾配が `accum_steps` 倍に膨らみ、学習率を勝手に何倍にもしたのと同じになってしまいます。また、`opt.step()` と `opt.zero_grad()` は「`accum_steps` 回たまった後に1回だけ」呼びます。

</details>

#### 📝 確認問題（`ch16-q10` · Lv.3 · 勾配累積で損失を accum_steps で割る理由と zero_grad() のタイミング）

マイクロバッチサイズ `8` で `accum_steps = 4` 回の**勾配累積**を行い、実質バッチサイズ `32` の学習を再現します。もし誤って損失を `4` で割らずに `loss.backward()` を4回呼んでから `opt.step()` すると、何が起きますか？

- **a.** 毎回 `.grad` が新しい値で上書きされるため、最初の3回分が捨てられて4回目だけで更新されてしまう
- **b.** `.grad` に4回分が加算されて本来の4倍の大きさになり、学習率を4倍にしたのと同様に不安定化する
- **c.** PyTorchが累積回数を自動検出して平均化するため、バッチ32一括と数学的に完全に同じ勾配になる
- **d.** ミニバッチが4分割された影響で勾配の大きさが本来の4分の1に縮み、学習がほとんど進まなくなる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：PyTorchで `zero_grad()` を呼ばずに `backward()` を繰り返すと、`.grad` には勾配が加算されていきます。
- **ヒント 2**：8件ずつの「平均損失」を4回そのまま足し合わせると、32件全体の「平均損失」の何倍の大きさになるか計算してみましょう。
- **正解**：`b`

**考え方**：32件全体の平均損失は $\frac{1}{4}(\bar{L}_1 + \bar{L}_2 + \bar{L}_3 + \bar{L}_4)$ です。各マイクロバッチで $\bar{L}_i / 4$ にしてから `backward()` することで初めて、バッチ32一括と数学的に完全一致します。

1. バッチ32の平均損失 $= \frac{1}{32}\sum_{j=1}^{32} \ell_j$。
1. バッチ8の平均を4つに分けると $= \frac{1}{4}\left(\frac{1}{8}\sum_{1..8}\ell_j + \dots + \frac{1}{8}\sum_{25..32}\ell_j\right)$。
1. したがって各回 `loss / 4` を `backward()` し、4回たまった後に `opt.step()` と `opt.zero_grad()` を1回だけ呼ぶ。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q10

</details>

#### 📝 確認問題（`ch16-q11` · Lv.3 · 勾配累積の平均と全体平均の一致を純Pythonで確認）

8個の損失値を「一括で平均」した場合と、「2個ずつ4回に分けて `4` で割りながら累積」した場合を比べます。次のコードの出力を予測してください。

```python
losses = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0]
full_mean = sum(losses) / len(losses)

accum = 0.0
for i in range(4):
    chunk = losses[i*2:(i+1)*2]
    accum += (sum(chunk) / len(chunk)) / 4

print(full_mean, accum, full_mean == accum)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：8要素全体の平均 `full_mean` と、2要素ずつ4回に分けて `4` で割りながら足した `accum` の関係を確かめます。
- **ヒント 2**：`losses` の全8要素の合計を `8` で割って `full_mean` を求め、4つのチャンク平均をそれぞれ `4` で割って足した値と一致するか確認しましょう。
- **期待される出力**：`9.0 9.0 True`

**考え方**：各チャンクの平均 `(3.0, 7.0, 11.0, 15.0)` をそれぞれ `4` で割って足すと `(3+7+11+15)/4 = 36/4 = 9.0` となり、一括平均 `9.0` と完全に一致します。

1. 一括平均：`(2+4+6+8+10+12+14+16) / 8 = 72 / 8 = 9.0`。
1. 4つのチャンク平均：`3.0, 7.0, 11.0, 15.0`。
1. それぞれを4で割って足す：`(3.0 + 7.0 + 11.0 + 15.0) / 4 = 9.0` → `9.0 9.0 True`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q11

</details>

## 学習の工学2：活性化の再計算

学習中のメモリを一番食うのは、パラメータではなく、**逆伝播のために取っておく途中の計算結果**（活性化）であることがよくあります。第7章の計算グラフを思い出してください。`backward()` のために、順方向の途中の値を全部覚えておく必要がありました。

> **💡 理論のポイント：勾配チェックポイント**
>
> **勾配チェックポイント**（activation checkpointing）は、途中の値を**覚えておかずに捨ててしまい**、逆伝播で必要になったときに**もう一度計算し直す**方法です。計算時間が約3割増える代わりに、メモリを大幅に節約できます。「メモを取らずに、必要なときに考え直す」作戦です。PyTorchでは `torch.utils.checkpoint.checkpoint(block, x)` のように、ブロックを包むだけで使えます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from torch.utils.checkpoint import checkpoint

block = nn.Sequential(nn.Linear(64, 256), nn.GELU(), nn.Linear(256, 64))
x = torch.randn(8, 64, requires_grad=True)
out1 = block(x).sum(); out1.backward(); g1 = x.grad.clone()
x.grad = None
out2 = checkpoint(block, x, use_reentrant=False).sum(); out2.backward(); g2 = x.grad
print("同じ勾配？", torch.allclose(g1, g2))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
同じ勾配？ True
```

</details>

<details><summary><b>基礎の補足：活性化の再計算：何を捨てて、何を残すのか？</b>（クリックで開く）</summary>

Transformerの24層すべての中間計算（LayerNorm後・Q/K/V・Attentionスコア・MLPの4倍幅の中間層など）をVRAMに残すと、長文では重み本体より何倍もメモリを食います。

| 方式 | 順伝播でVRAMに保存するもの | 逆伝播の動き | 計算時間 | 勾配の精度 |
|---|---|---|---|---|
| 通常学習 | 全ブロック内部の中間値すべて | 保存した値をそのまま読む | 基準（1.0倍） | 正確 |
| **勾配チェックポイント** | **各ブロックの入口 `x` だけ** | 必要な瞬間にそのブロックだけ再計算 | 約1.3倍（+3割） | **数学的に完全一致** |

推論時（`torch.no_grad()`）はそもそも逆伝播をしないため、勾配チェックポイントは**学習専用の省メモリ技**です。

</details>

#### 📝 確認問題（`ch16-q12` · Lv.2 · 活性化の再計算（勾配チェックポイント）のトレードオフ）

**勾配チェックポイント（activation checkpointing）** の仕組みとトレードオフとして正しいものはどれですか？

- **a.** モデルの重みテンソルを学習中にディスクへ退避させることで、推論時のトークン生成速度を5倍に高める技術
- **b.** 順伝播の活性化を捨て逆伝播で再計算することで、時間は約3割増えるが同じ勾配のままメモリを節約できる
- **c.** 学習途中のモデル重みを一定ステップごとにストレージへ保存しておく、障害復旧用チェックポイント機能のこと
- **d.** 途中のTransformer層の逆伝播を丸ごと省略して近似勾配を使うため、計算は2倍速くなるが精度が落ちる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「メモを取って机を埋める代わりに、必要になったときに考え直す」というトレードオフを思い出しましょう。
- **ヒント 2**：中間値（活性化）をVRAMに残さず逆伝播で再計算すると、メモリ・計算時間・勾配の正確さはそれぞれどうなるか整理しましょう。
- **正解**：`b`

**考え方**：「計算時間（約1.3倍）」と引き換えに「活性化メモリ」を劇的に減らす、時間とメモリの典型的なトレードオフです。

1. 通常：全ブロックの途中の値をVRAMに保存 → 長文や深いモデルでVRAMがパンク。
1. チェックポイント：各ブロックの入口だけ覚え、内部の中間値は捨てる。
1. 逆伝播時：そのブロックだけ順伝播をやり直して中間値を復元し、正確な勾配を計算する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q12

</details>

## 学習の工学3：FlashAttention

Attentionは、文章の長さ $T$ に対して $T \times T$ の点数表を作ります。長さ8000なら6400万マス。これをGPUのメモリに書いて、また読んで……をくり返すのが遅さの原因でした。

**FlashAttention** は、点数表を**小さなタイルに分けて**、GPUの中の超高速な小さいメモリの中だけで計算を済ませてしまう方法です。結果は同じなのに、速く、メモリも少なく済みます。PyTorchでは `F.scaled_dot_product_attention` を使うと、使える環境なら自動でFlashAttention（などの高速な方法）が選ばれます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
q, k, v = (torch.randn(2, 4, 128, 32) for _ in range(3))     # [B, ヘッド, T, hs]

att = q @ k.transpose(-2, -1) / math.sqrt(32)
att = att.masked_fill(torch.tril(torch.ones(128, 128)) == 0, float("-inf"))
manual = F.softmax(att, dim=-1) @ v

fast = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("手作りと同じ？", torch.allclose(manual, fast, atol=1e-5))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
手作りと同じ？ True
```

</details>

第15章の `CausalSelfAttention` の中の4行（点数→マスク→softmax→Vを掛ける）を、この1行に置き換えるだけで、GPUでは何倍も速くなります。自分のGPTを速くしたくなったら、最初に試すべき改造です。

<details><summary><b>基礎の補足：GPUのメモリ階層（SRAM vs HBM）：FlashAttentionが「近似なし」で速く省メモリな理由</b>（クリックで開く）</summary>

GPUの内部には、性格のまったく違う2種類のメモリがあります。

| メモリ | 例え | 容量の目安 | 読み書きの速さ |
|---|---|---|---|
| **HBM**（メインVRAM） | 広い書庫 | 数十GB（T4は約15GB） | 比較的遅い（ここへの往復がボトルネック！） |
| **SRAM**（チップ内キャッシュ） | 机の上 | 数十MB（ごく小さい） | **HBMの10倍以上 超高速！** |

- **標準Attention**：巨大な $T \times T$ 行列（$O(T^2)$）を丸ごとHBMに書き出し → マスクのためにまたHBMから読んで書き出し → softmaxのためにまた読み書き……と、遅い書庫を何度も往復します。
- **FlashAttention**：$Q, K, V$ を小さなタイルに分けて机の上（SRAM）へ運び、オンラインsoftmaxという工夫で**巨大な $T \times T$ 行列を一度もHBMに書き出さずに**正確な出力だけを計算します。その結果、HBMの消費は $O(T^2) \to O(T)$ に激減し、速度も数倍に跳ね上がります。

</details>

<details><summary><b>もっと深く：torch.compile</b>（クリックで開く）</summary>

`model = torch.compile(model)` の1行で、PyTorchがモデルの計算をまとめて最適化してくれます（初回の実行は準備に時間がかかります）。GPU上では、学習が数割速くなることがよくあります。

</details>

#### 📝 確認問題（`ch16-q13` · Lv.3 · FlashAttentionが速く省メモリな理由（HBM vs SRAMのメモリ階層））

**FlashAttention** が、通常のAttentionと**数学的に同じ結果**を出しながらずっと速く・省メモリになる本質的な理由は何ですか？

- **a.** 計算の重い指数関数やsoftmaxをすべて廃止し、単純な線形加算でAttention重みを代用するから
- **b.** $T \times T$ 行列を遅いHBMへ書き出さず、小タイルに分けて高速なSRAM上で計算を完結させるから
- **c.** 長文のAttention計算だけをネットワーク経由で別サーバーのCPU群へ自動分散して並列処理するから
- **d.** 過去トークンの約9割をランダムに間引いて無視し、総演算回数自体を10分の1へ削る近似手法だから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：GPUのボトルネックは「計算ユニットの速さ」よりも、メインメモリ（HBM）とのデータ往復にあることを思い出しましょう。
- **ヒント 2**：標準Attentionは $T \times T$ 行列をHBMへ何度も読み書きします。FlashAttentionはチップ内の超高速メモリ（SRAM）をどう活用するでしょうか？
- **正解**：`b`

**考え方**：標準Attentionは $Q K^\top$・マスク・softmax のたびに巨大な $T \times T$ 行列をHBMへ読み書きしますが、FlashAttentionはタイルごとにSRAM内で一気に処理して出力だけをHBMへ書き戻すため、メモリ使用量も $O(T^2)$ から $O(T)$ へ激減します。

1. GPUには「大容量だが遅い **HBM**（VRAM）」と「極小（数十MB）だが超高速な **SRAM**（チップ内キャッシュ）」がある。
1. 標準Attentionは $T \times T$ 行列をHBMに丸ごと作って何度も読み書きするため遅くメモリを食う。
1. FlashAttentionは $Q, K, V$ を小さなブロック（タイル）に分けてSRAMへ読み込み、オンラインsoftmaxで $T \times T$ をHBMに作らずに同じ答えを求める。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q13

</details>

## 何百枚ものGPUで学習する：分散学習

1枚のGPUに乗らない、あるいは1枚では何年もかかる学習は、たくさんのGPUで分担します。この教科書の範囲（無料の環境）ではめったに使いませんが、論文やニュースを読むために、考え方だけは押さえておきましょう。

> **💡 理論のポイント：分散学習の4つの作戦**
>
> - **データ並列**（DDP）：GPUごとにモデルを**丸ごとコピー**し、違うデータで学習。勾配だけを全GPUで平均する。一番シンプルで、よく使われる。でも、モデルが1枚に乗らないと使えない
> - **ZeRO / FSDP**：データ並列の発展版。パラメータ・勾配・Adamの記録（第15章の16バイトの内訳）を、GPUたちで**分割して持ち合う**。必要なときだけ集めてくる。1枚に乗らない大きさのモデルも学習できる
> - **テンソル並列**：1つの行列の掛け算を、**行列を分割して**複数のGPUで計算する。1つの層が大きすぎるときに使う。GPU間の通信が多いので、同じマシン内のGPUどうしで使う
> - **パイプライン並列**：層ごとに担当のGPUを決める（1〜10層はGPU1、11〜20層はGPU2…）。工場の流れ作業。待ち時間（バブル）を減らす工夫が必要
>
> 最先端のLLMは、これらを全部組み合わせて（「3D並列」などと呼ばれます）、数千〜数万枚のGPUで学習されています。

> **¥0 無料枠のコツ：0円で分散学習を体験するなら**
>
> Kaggleのノートブックでは、無料で「T4×2」（GPU2枚）を選べることがあります。PyTorchの `torchrun` と DDP を使えば、2枚でのデータ並列を体験できます。興味がある人は、PyTorch公式の「DDPチュートリアル」を、Kaggleで試してみてください（[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)にKaggleの使い方があります）。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **KVキャッシュ付きの生成で結果が変わる**：位置の埋め込みの番号（`start_pos`）がずれていないか確認しましょう。キャッシュを使うときは、新しいトークンの位置は「これまでの長さ」から始まります。
> - **勾配累積で学習がおかしい**：損失を累積回数で割っていないと、学習率が実質何倍にもなってしまいます。また、`zero_grad()` は「更新した後」に1回だけ呼びます。
> - **`scaled_dot_product_attention` が遅い・使えない**：古いGPUや古いPyTorchでは、FlashAttentionが選ばれないことがあります。それでも結果は正しいので、心配はいりません。

#### 📝 確認問題（`ch16-q14` · Lv.3 · 分散学習の4方式（DDP・ZeRO/FSDP・テンソル並列・パイプライン並列））

巨大なLLMの学習では1枚のGPUにモデル全体（重み・勾配・AdamW状態）が乗り切りません。モデルの重み・勾配・AdamWの状態を複数GPUで**分割して持ち合い**、必要な層を計算するときだけ集めてくる分散学習の方式はどれですか？

- **a.** **KVキャッシュ**（過去のKey・Valueテンソル再利用）
- **b.** **ZeRO / FSDP**（Fully Sharded Data Parallel）
- **c.** **top-p**（Nucleus Sampling：累積確率サンプリング）
- **d.** 単純な**データ並列**（DDP: Distributed Data Parallel）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：データ並列では全GPUに同じ重み・勾配・Adam状態が重複します。この重複を解消してGPU間で「分割（シャード）」する方式です。
- **ヒント 2**：「Fully Sharded Data Parallel」という名前のとおり、重み・勾配・オプティマイザ状態を分割保持し必要時だけ集める方式を選びましょう。
- **正解**：`b`

**考え方**：DDPでは各GPUに同じコピーが重複して無駄が生じますが、ZeRO / FSDPはその重複をなくしてGPU間で分割保持することで、1枚に収まらない巨大モデルの学習を可能にします。

1. DDP：各GPUにモデル丸ごとコピー＋違うデータを担当。
1. ZeRO / FSDP：重み・勾配・Adam状態をGPU間で分割保持（シャード）し、必要時に通信で集める。
1. テンソル並列：1つの行列積自体を複数GPUに分割。
1. パイプライン並列：1〜10層をGPU1、11〜20層をGPU2のように層ごとに分担。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html#ex-ch16-q14

</details>

**✅ できたらチェック**

- [ ] KVキャッシュあり・なしで、同じ結果がより速く出ることを確かめた
- [ ] KVキャッシュのメモリを計算し、GQAで減る理由を説明できる
- [ ] top-k と top-p の違いを説明できる
- [ ] 勾配累積で、大きなバッチと同じ勾配になることを確かめた
- [ ] `scaled_dot_product_attention` が手作りのAttentionと一致することを確かめた
- [ ] 分散学習の4つの作戦を、ひとことずつ説明できる

> **📝 まとめ**
>
> - **KVキャッシュ**：過去のKey・Valueを取っておき、新しいトークンだけ計算する。生成が大幅に速くなる
> - KVキャッシュのメモリ ＝ 2 × 層数 × K/Vのヘッド数 × ヘッドの大きさ × トークン数 × バイト数。**GQA**で減らせる
> - **top-k / top-p** で候補を絞ってからサンプリングする。くり返しペナルティも
> - **勾配累積**・**勾配チェックポイント**・**FlashAttention**・混合精度で、少ないメモリで速く学習する
> - 大きなLLMは、データ並列・ZeRO・テンソル並列・パイプライン並列を組み合わせて学習する

**ボンゴ先生**：第4部、お疲れさま！ ここまでで、キミは「LLMを中身から作れる人」になった。次の第5部からは、視点を変える。世界中の企業が何億円もかけて事前学習したモデルを**借りてきて**、キミの目的に合わせて鍛え直す。現代のLLM開発の、いちばん実践的な部分だ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html
- **次の章**：[第17章 Hugging Faceと事前学習済みモデル](https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html)